# 10 - Métricas Pix

Consolida métricas de negócio, regressão e classificação.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
import pandas as pd
from pyspark.sql import functions as F

from src.config import PIX_FEE_SAVINGS_DIR, PIX_MONTHLY_INDICATORS_DIR, PIX_TRANSFER_SAVINGS_DIR, REPORTS_DIR, create_project_directories
from src.spark_session import get_spark_session

create_project_directories(False)
spark = get_spark_session("10-metrics-pix")

In [ ]:
monthly = spark.read.parquet(str(PIX_MONTHLY_INDICATORS_DIR))
card = spark.read.parquet(str(PIX_FEE_SAVINGS_DIR))
transfer = spark.read.parquet(str(PIX_TRANSFER_SAVINGS_DIR))
agg = monthly.agg(
    F.sum("quantidade_transacoes").alias("total_transacoes"),
    F.sum("valor_total").alias("valor_total_movimentado"),
    (F.sum("valor_total") / F.sum("quantidade_transacoes")).alias("ticket_medio_geral"),
).first().asDict()
maior_qtd = monthly.orderBy(F.desc("quantidade_transacoes")).first()["ano_mes"]
maior_valor = monthly.orderBy(F.desc("valor_total")).first()["ano_mes"]
rows = [("total_transacoes", agg["total_transacoes"]), ("valor_total_movimentado", agg["valor_total_movimentado"]), ("ticket_medio_geral", agg["ticket_medio_geral"]), ("maior_mes_quantidade", maior_qtd), ("maior_mes_valor", maior_valor), ("economia_estimativa_total_cartao", card.agg(F.sum("economia_estimada")).first()[0]), ("economia_estimativa_total_transferencia", transfer.agg(F.sum("economia_estimada")).first()[0])]
pd.DataFrame(rows, columns=["metrica", "valor"]).to_csv(REPORTS_DIR / "business_metrics_summary.csv", index=False)

frames = []
for name in ["regression_metrics.csv", "classification_metrics.csv", "business_metrics_summary.csv"]:
    path = REPORTS_DIR / name
    if path.exists():
        df = pd.read_csv(path)
        df.insert(0, "origem", name)
        frames.append(df)
if frames:
    pd.concat(frames, ignore_index=True).to_csv(REPORTS_DIR / "model_metrics_summary.csv", index=False)
print("Métricas consolidadas em reports/.")

In [ ]:
spark.stop()